# Train YOLOv8 on AVOS (Colab)

Fine-tunes a COCO-pretrained YOLOv8 checkpoint on your already-prepared
AVOS train/val split. This notebook does not create or modify your split --
it expects a standard Ultralytics YOLO `data.yaml` (`train:`, `val:`,
`names:`) pointing at it.

**Before running:** Runtime menu -> Change runtime type -> GPU.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
!git clone https://github.com/cxia0024/hypospadias-object-detection.git
%cd hypospadias-object-detection
!git checkout claude/surgical-phase-recognition-wqrp7r

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Sanity check: run the unit tests

In [ ]:
!python -m pytest tests/ -q

## 5. Mount Google Drive

Your prepared AVOS train/val split (images + YOLO-format label .txt files)
and its `data.yaml` should already live in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 6. Validate your data.yaml

Edit `DATA_YAML` to point at your split's config, then this cell checks it has
`train`/`val`/`names` and that the split paths actually exist -- before
spending GPU time on a typo.

In [ ]:
import sys
sys.path.insert(0, "src")
from stage1_detection.train import validate_data_yaml

DRIVE_ROOT = "/content/drive/MyDrive/hypospadias"  # <-- change to your Drive folder
DATA_YAML = f"{DRIVE_ROOT}/data/avos/data.yaml"     # <-- change to your split's data.yaml

config = validate_data_yaml(DATA_YAML)
print(f"OK. Training on {len(config['names'])} classes: {config['names']}")

## 7. Train

`yolov8s.pt` is the COCO-pretrained starting point per the Methods.
Swap to `yolov8n.pt` for a faster/smaller run, or `yolov8m.pt`/`yolov8l.pt`
for more capacity if the GPU has the memory for it.

In [ ]:
from stage1_detection.train import train_yolo, find_best_checkpoint

PROJECT = "runs/train"
NAME = "avos_yolov8"

train_yolo(
    data=DATA_YAML,
    model="yolov8s.pt",
    epochs=100,
    imgsz=640,
    batch=16,
    project=PROJECT,
    name=NAME,
    seed=0,
)

best = find_best_checkpoint(PROJECT, NAME)
print(f"Best checkpoint: {best}")

## 8. Save the checkpoint to Drive

This is the file `configs/stage1_datasets.yaml`'s `model_path` should point
at for the Stage 1 zero-shot evaluation (see `run_stage1_eval_colab.ipynb`).

In [ ]:
import shutil
from pathlib import Path

out_path = Path(f"{DRIVE_ROOT}/models/yolov8_avos_best.pt")
out_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, out_path)
print(f"Saved to {out_path}")